# Milestone 1 — Member B (VAE Lead) Step-by-Step Colab Notebook
Run each cell top-to-bottom on a **Colab T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).

- Checkpoints are saved after **every epoch** to Google Drive so if Colab disconnects, re-running resumes automatically without losing progress.
- `TEST` is never touched; all evaluation and model selection use `VAL` only.

In [ ]:
# Step 1: Mount Google Drive & verify T4 GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 2: Unpack CI_Project_Colab_Ready.zip (if needed) & set paths
import os, zipfile
ZIP_ON_DRIVE = '/content/drive/MyDrive/CI_Project_Colab_Ready.zip'
REPO_DIR = '/content/drive/MyDrive/CI_Project'
CKPT_DIR = '/content/drive/MyDrive/CI_Checkpoints_M1'
MEMBER_NAME = 'Bola'

# Fast tip: extract the 12,653 images to /content/local_repo for instant disk speed, while keeping results on Drive!
LOCAL_REPO = '/content/CI_Project'
if not os.path.exists(LOCAL_REPO):
    assert os.path.exists(ZIP_ON_DRIVE), f'Please upload CI_Project_Colab_Ready.zip to MyDrive first! Looked at: {ZIP_ON_DRIVE}'
    print('Extracting CI_Project_Colab_Ready.zip to fast Colab SSD (takes ~3s)...')
    with zipfile.ZipFile(ZIP_ON_DRIVE, 'r') as zf:
        zf.extractall(LOCAL_REPO)

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(f'{REPO_DIR}/results', exist_ok=True)
%cd {LOCAL_REPO}
os.environ['PYTHONPATH'] = f'{LOCAL_REPO}/src'

In [ ]:
# Step 3: Build dataset.npz, copy to fast local /content disk, and run all unit tests
!python3 scripts/build_dataset.py --splits-dir T12_Splits --output-dir results/data
!cp results/data/dataset.npz /content/dataset.npz
!PYTHONPATH=src python3 -m unittest discover -s tests -v

In [ ]:
# Step 4: Quick 1-epoch smoke test on dummy data (takes ~5 seconds — verifies zero bugs!)
!PYTHONPATH=src python3 -m vae.train --name smoke --dummy --epochs 1 --results-dir /tmp/smoke_res

In [ ]:
# Step 5: Run ALL 6 controlled VAE experiments (same 40-epoch budget, checkpoints & results saved on Drive!)
!PYTHONPATH=src python3 -m vae.run_study \
    --member "{MEMBER_NAME}" \
    --data /content/dataset.npz \
    --results-dir "{REPO_DIR}/results" \
    --ckpt-dir "{CKPT_DIR}"

In [ ]:
# Step 6: Inspect Table 1.2, selected VAE, and generated plots!
from IPython.display import Image, Markdown, display
import json

display(Markdown(open(f'{REPO_DIR}/results/table_1_2.md').read()))
sel = json.load(open(f'{REPO_DIR}/results/selected_vae.json'))
print('Chosen VAE summary:', json.dumps(sel, indent=2))

display(Image(f'{REPO_DIR}/results/figures/vae_training_curves.png'))
display(Image(f"{REPO_DIR}/results/figures/vae_recon_{sel['selected_run']}.png"))

In [ ]:
# Step 7 (Oral Exam Reproducibility Check): Reproduce exact VAL numbers from the chosen checkpoint
import json
sel = json.load(open(f'{REPO_DIR}/results/selected_vae.json'))
ckpt = sel['checkpoint_path']
!PYTHONPATH=src python3 -m vae.evaluate --checkpoint "{ckpt}" --data /content/dataset.npz